In [3]:

import os

base = "/data/workspace/61448ef7-eb83-45fe-a53b-e87021227cc1/mf_atom_detection"

# ── mf_detector.py ──────────────────────────────────────────────────────────
mf_detector = '''"""
mf_detector.py  --  Vectorized Matched Filter Array core.

For K sites and an image frame, compute:
    scores[k] = dot(roi_k.flat, h_k.flat)          (raw MF score)
    p_mf[k]   = sigmoid((scores[k] - bias) / scale) (calibrated probability)

The MF is the SNR-optimal linear detector for a known signal in additive
white Gaussian noise (Neyman-Pearson lemma).
"""

import numpy as np
from numpy.typing import NDArray


def extract_rois(frame: NDArray[np.float32],
                 site_centers: NDArray[np.int32],
                 roi_size: int,
                 roi_offsets: NDArray[np.float32] | None = None
                 ) -> NDArray[np.float32]:
    """
    Extract K ROI patches from a full fluorescence frame.

    Parameters
    ----------
    frame        : (H, W) float32 image (photoelectron counts)
    site_centers : (K, 2) int32 array of [row, col] site centers
    roi_size     : patch side length (pixels), must be odd
    roi_offsets  : (K, 2) float32 [dy, dx] drift correction (sub-pixel, rounded)

    Returns
    -------
    rois : (K, roi_size, roi_size) float32
    """
    K = site_centers.shape[0]
    half = roi_size // 2
    H, W = frame.shape
    rois = np.zeros((K, roi_size, roi_size), dtype=np.float32)

    for k in range(K):
        r0, c0 = int(site_centers[k, 0]), int(site_centers[k, 1])
        if roi_offsets is not None:
            r0 += int(round(roi_offsets[k, 0]))
            c0 += int(round(roi_offsets[k, 1]))
        r_lo = max(r0 - half, 0)
        r_hi = min(r0 + half + 1, H)
        c_lo = max(c0 - half, 0)
        c_hi = min(c0 + half + 1, W)
        # copy into output patch (handles boundary sites)
        dr = r_lo - (r0 - half)
        dc = c_lo - (c0 - half)
        patch_h = r_hi - r_lo
        patch_w = c_hi - c_lo
        rois[k, dr:dr+patch_h, dc:dc+patch_w] = frame[r_lo:r_hi, c_lo:c_hi]
    return rois


def compute_mf_scores(rois: NDArray[np.float32],
                      templates: NDArray[np.float32]) -> NDArray[np.float32]:
    """
    Vectorized MF score computation.

    Parameters
    ----------
    rois      : (K, R, R) float32 -- extracted ROI patches
    templates : (K, R, R) float32 -- unit-norm Gaussian templates

    Returns
    -------
    scores : (K,) float32  -- raw matched-filter output per site
    """
    # scores[k] = sum_{i,j} rois[k,i,j] * templates[k,i,j]
    # Vectorized as batch inner product: reshape to (K, R*R), then row-wise dot
    K = rois.shape[0]
    R2 = rois.shape[1] * rois.shape[2]
    scores = (rois.reshape(K, R2) * templates.reshape(K, R2)).sum(axis=1)
    return scores


def scores_to_probs(scores: NDArray[np.float32],
                    mu_atom: float, sigma_atom: float,
                    mu_bg: float,   sigma_bg: float) -> NDArray[np.float32]:
    """
    Convert raw MF scores to posterior probabilities via Bayesian LRT.

    Assumes equal prior P(H1) = P(H0) = 0.5 and Gaussian score distributions:
        score | H1 ~ N(mu_atom, sigma_atom^2)
        score | H0 ~ N(mu_bg,   sigma_bg^2)

    LLR = log p(s|H1) - log p(s|H0)
    p_post = sigmoid(LLR)

    Parameters
    ----------
    scores     : (K,) float32
    mu_atom, sigma_atom : H1 Gaussian parameters
    mu_bg,   sigma_bg   : H0 Gaussian parameters

    Returns
    -------
    probs : (K,) float32 in (0, 1)
    """
    # log p(s|H1) - log p(s|H0)
    llr = (
        -0.5 * ((scores - mu_atom) / sigma_atom) ** 2
        + 0.5 * ((scores - mu_bg)  / sigma_bg)  ** 2
        + np.log(sigma_bg / sigma_atom)
    )
    probs = 1.0 / (1.0 + np.exp(-llr.astype(np.float32)))
    return probs.astype(np.float32)


def calibrate_mf_params(rois_atom: NDArray[np.float32],
                        rois_bg:   NDArray[np.float32],
                        templates: NDArray[np.float32]
                        ) -> tuple[float, float, float, float]:
    """
    Estimate MF score distribution parameters from calibration ROI sets.

    Parameters
    ----------
    rois_atom : (N, R, R) -- ROIs confirmed to contain atoms
    rois_bg   : (N, R, R) -- ROIs confirmed to be empty
    templates : (K, R, R) -- templates (use mean template if K>1 for simplicity)

    Returns
    -------
    (mu_atom, sigma_atom, mu_bg, sigma_bg)
    """
    # Use a single representative template (mean, re-normalized)
    tmpl = templates.mean(axis=0)
    norm = np.linalg.norm(tmpl)
    if norm > 0:
        tmpl = tmpl / norm
    tmpl_rep = tmpl[np.newaxis, :, :]   # (1, R, R)

    # broadcast templates
    N_atom = rois_atom.shape[0]
    N_bg   = rois_bg.shape[0]
    t_atom = np.broadcast_to(tmpl_rep, (N_atom,) + tmpl.shape)
    t_bg   = np.broadcast_to(tmpl_rep, (N_bg,)   + tmpl.shape)

    s_atom = compute_mf_scores(rois_atom, t_atom)
    s_bg   = compute_mf_scores(rois_bg,   t_bg)

    return (float(s_atom.mean()), float(s_atom.std()) + 1e-6,
            float(s_bg.mean()),   float(s_bg.std())   + 1e-6)
'''

# ── layer1_mf.py ─────────────────────────────────────────────────────────────
layer1_mf = '''"""
layer1_mf.py  --  L1: Matched Filter fast-decision layer.

Replaces the lightweight CNN with a Matched Filter array.
Retains Bayesian log-odds temporal fusion and CUSUM loss detection.
L1.5 (physical diagnostic router) is REMOVED.

Decision logic per site k at frame t:
  1. Compute MF score s(k,t) and convert to probability p_mf(k,t)
  2. Bayesian log-odds update:
       x(k,t) = x(k,t-1) + log(ps/(1-ps)) + lambda * log(p_mf/(1-p_mf))
       p_post(k,t) = sigmoid(x(k,t))
  3. CUSUM update:
       S(k,t) = max(0, S(k,t-1) + log((1-p_mf)/p_mf))
     If S(k,t) > h_loss  -> ERASURE_LOSS
  4. Double-threshold decision:
       p_post > theta_H  -> ATOM_PRESENT
       p_post < theta_L  -> ATOM_ABSENT
       otherwise         -> route to L2
"""

import numpy as np
from numpy.typing import NDArray
from enum import IntEnum


class Decision(IntEnum):
    ATOM_PRESENT  = 1
    ATOM_ABSENT   = 0
    ROUTE_L2      = -1
    ERASURE_LOSS  = 2
    CORR_LOSS     = 3    # kept for interface compatibility; never raised without L1.5
    DRIFT_ALARM   = 4


class L1State:
    """Per-site temporal state for Bayesian fusion and CUSUM."""

    def __init__(self, n_sites: int):
        self.n_sites = n_sites
        # Bayesian log-odds state (initialised neutral)
        self.log_odds: NDArray[np.float32] = np.zeros(n_sites, dtype=np.float32)
        # CUSUM statistics
        self.S_cusum:  NDArray[np.float32] = np.zeros(n_sites, dtype=np.float32)
        # frame counter for warmup
        self.frame_count: int = 0

    def reset(self) -> None:
        self.log_odds[:] = 0.0
        self.S_cusum[:]  = 0.0
        self.frame_count  = 0


def _safe_logit(p: NDArray[np.float32], eps: float = 1e-7) -> NDArray[np.float32]:
    p = np.clip(p, eps, 1.0 - eps)
    return np.log(p / (1.0 - p))


def run_l1(p_mf:     NDArray[np.float32],
           state:    L1State,
           theta_H:  float,
           theta_L:  float,
           p_survival: float,
           lambda_obs: float,
           h_loss:   float,
           h_corr:   float,
           warmup:   int
           ) -> tuple[NDArray[np.int8], NDArray[np.float32]]:
    """
    Run one frame through L1.

    Parameters
    ----------
    p_mf        : (K,) float32 -- MF-derived per-site probability
    state       : L1State (mutated in-place)
    theta_H/L   : double-threshold bounds
    p_survival  : atom frame-to-frame survival probability
    lambda_obs  : Bayesian observation weight
    h_loss      : CUSUM erasure threshold
    h_corr      : CUSUM correlated-loss threshold (unused without L1.5)
    warmup      : frames before temporal fusion activates

    Returns
    -------
    decisions : (K,) int8   -- Decision enum value per site
    p_post    : (K,) float32 -- posterior probability per site
    """
    K = state.n_sites
    state.frame_count += 1
    use_temporal = state.frame_count > warmup

    # -- CUSUM update (always; needed for ERASURE detection)
    cusum_increment = _safe_logit(1.0 - p_mf)   # log((1-p)/p)  positive when no atom
    state.S_cusum = np.maximum(0.0, state.S_cusum + cusum_increment)

    # -- Bayesian log-odds update
    survival_logit = float(np.log(p_survival / (1.0 - p_survival)))
    obs_logit = lambda_obs * _safe_logit(p_mf)

    if use_temporal:
        state.log_odds = state.log_odds + survival_logit + obs_logit
    else:
        state.log_odds = _safe_logit(p_mf)   # single-frame fallback

    p_post = 1.0 / (1.0 + np.exp(-state.log_odds)).astype(np.float32)
    p_post = np.clip(p_post, 1e-7, 1.0 - 1e-7).astype(np.float32)

    # -- Decision
    decisions = np.full(K, Decision.ROUTE_L2, dtype=np.int8)

    # ERASURE has highest priority
    erasure_mask = state.S_cusum > h_loss
    decisions[erasure_mask] = int(Decision.ERASURE_LOSS)
    state.S_cusum[erasure_mask] = 0.0   # reset after trigger

    # High-confidence accept / reject
    accept_mask = (~erasure_mask) & (p_post >= theta_H)
    reject_mask = (~erasure_mask) & (p_post <= theta_L)
    decisions[accept_mask] = int(Decision.ATOM_PRESENT)
    decisions[reject_mask] = int(Decision.ATOM_ABSENT)

    return decisions, p_post
'''

# ── layer2_mf.py ─────────────────────────────────────────────────────────────
layer2_mf = '''"""
layer2_mf.py  --  L2: Matched Filter precise-judgment layer.

Applied only to sites that L1 routes to L2 (decisions == ROUTE_L2).
Uses a longer exposure (higher photon count) for definitive judgment.
"""

import numpy as np
from numpy.typing import NDArray
from src.layer1_mf import Decision


def run_l2(p_mf_l2:   NDArray[np.float32],
           l2_indices: NDArray[np.int64],
           threshold:  float = 0.50
           ) -> NDArray[np.int8]:
    """
    Final adjudication for routed sites.

    Parameters
    ----------
    p_mf_l2    : (M,) float32 -- MF probability from long-exposure ROI
    l2_indices : (M,) int64   -- site indices that were routed to L2
    threshold  : decision threshold

    Returns
    -------
    decisions : (M,) int8 -- ATOM_PRESENT or ATOM_ABSENT
    """
    decisions = np.where(
        p_mf_l2 >= threshold,
        np.int8(Decision.ATOM_PRESENT),
        np.int8(Decision.ATOM_ABSENT)
    ).astype(np.int8)
    return decisions
'''

# ── system.py ────────────────────────────────────────────────────────────────
system_py = '''"""
system.py  --  MF-Array two-layer detection system (PA-3HDA-MF v1.0).

Architecture (L1.5 removed):
    L0  : Drift awareness (ROI offset correction, background thread -- stubbed here)
    L1  : MF-Array fast layer + Bayesian temporal fusion + CUSUM
    L2  : MF-Array precise layer (long exposure)
    Out : 5-type structured output  {ATOM_PRESENT, ATOM_ABSENT, ERASURE_LOSS,
                                     CORR_LOSS, DRIFT_ALARM}
"""

import numpy as np
from numpy.typing import NDArray
from dataclasses import dataclass, field

from src.config import SystemConfig
from src.psf import build_template_array
from src.mf_detector import (compute_mf_scores, scores_to_probs,
                               extract_rois)
from src.layer1_mf import L1State, run_l1, Decision
from src.layer2_mf import run_l2


@dataclass
class FrameResult:
    """Per-frame detection result for all K sites."""
    decisions:   NDArray[np.int8]    # (K,)  Decision enum value
    p_post:      NDArray[np.float32] # (K,)  L1 posterior probability
    l2_routed:   NDArray[np.bool_]   # (K,)  mask of sites routed to L2
    n_erasure:   int = 0
    n_l2_routed: int = 0


class MFSystem:
    """Two-layer Matched Filter detection system."""

    def __init__(self, cfg: SystemConfig):
        self.cfg = cfg
        K = cfg.array.n_sites

        # Build site center coordinates
        rows = np.arange(cfg.array.n_rows)
        cols = np.arange(cfg.array.n_cols)
        rr, cc = np.meshgrid(rows, cols, indexing="ij")
        pitch = cfg.array.site_pitch_px
        margin = pitch
        self.site_centers = np.stack(
            [rr.ravel() * pitch + margin,
             cc.ravel() * pitch + margin], axis=1
        ).astype(np.int32)   # (K, 2)

        # Build MF templates
        self.roi_offsets: NDArray[np.float32] = np.zeros((K, 2), dtype=np.float32)
        self.templates = build_template_array(cfg.array, cfg.psf,
                                              roi_offsets=None)  # (K, R, R)

        # L1 state
        self.l1_state = L1State(K)

        # MF calibration params (set after calibrate() or from cfg.mf)
        self._mu_atom    = cfg.mf.mu_atom
        self._sigma_atom = cfg.mf.sigma_atom
        self._mu_bg      = cfg.mf.mu_bg
        self._sigma_bg   = cfg.mf.sigma_bg

    def set_mf_params(self, mu_atom: float, sigma_atom: float,
                      mu_bg: float, sigma_bg: float) -> None:
        self._mu_atom    = mu_atom
        self._sigma_atom = sigma_atom
        self._mu_bg      = mu_bg
        self._sigma_bg   = sigma_bg

    def process_frame(self,
                      frame_l1: NDArray[np.float32],
                      frame_l2: NDArray[np.float32] | None = None
                      ) -> FrameResult:
        """
        Process one imaging cycle.

        Parameters
        ----------
        frame_l1 : (H, W) float32  -- short-exposure fluorescence image
        frame_l2 : (H, W) float32 | None  --  long-exposure image
                   (only required for sites routed to L2; may be None if
                   the caller guarantees no L2 routing)

        Returns
        -------
        FrameResult
        """
        cfg = self.cfg

        # --- L1: extract ROIs and compute MF scores ---
        rois_l1 = extract_rois(frame_l1, self.site_centers,
                                cfg.array.roi_size, self.roi_offsets)
        scores_l1 = compute_mf_scores(rois_l1, self.templates)
        p_mf_l1   = scores_to_probs(scores_l1,
                                     self._mu_atom, self._sigma_atom,
                                     self._mu_bg,   self._sigma_bg)

        # --- L1: temporal fusion + CUSUM + threshold decision ---
        decisions, p_post = run_l1(
            p_mf        = p_mf_l1,
            state       = self.l1_state,
            theta_H     = cfg.l1.theta_H,
            theta_L     = cfg.l1.theta_L,
            p_survival  = cfg.physics.p_survival,
            lambda_obs  = cfg.l1.lambda_obs,
            h_loss      = cfg.cusum.h_loss,
            h_corr      = cfg.cusum.h_corr,
            warmup      = cfg.l1.warmup_frames,
        )

        # --- L2: precise MF for uncertain sites ---
        l2_mask = decisions == int(Decision.ROUTE_L2)
        n_l2    = int(l2_mask.sum())

        if n_l2 > 0 and frame_l2 is not None:
            l2_indices = np.where(l2_mask)[0].astype(np.int64)
            rois_l2 = extract_rois(frame_l2, self.site_centers[l2_indices],
                                   cfg.array.roi_size,
                                   self.roi_offsets[l2_indices])
            # Use same templates for L2 (longer exposure -> same PSF shape)
            scores_l2 = compute_mf_scores(rois_l2, self.templates[l2_indices])
            p_mf_l2   = scores_to_probs(scores_l2,
                                         self._mu_atom, self._sigma_atom,
                                         self._mu_bg,   self._sigma_bg)
            l2_decs = run_l2(p_mf_l2, l2_indices, threshold=cfg.l2.threshold)
            decisions[l2_indices] = l2_decs

        n_erasure = int((decisions == int(Decision.ERASURE_LOSS)).sum())

        return FrameResult(
            decisions   = decisions,
            p_post      = p_post,
            l2_routed   = l2_mask,
            n_erasure   = n_erasure,
            n_l2_routed = n_l2,
        )

    def reset(self) -> None:
        """Reset temporal state (call at start of each new atom loading cycle)."""
        self.l1_state.reset()
'''

# ── simulate.py ───────────────────────────────────────────────────────────────
simulate_py = '''"""
simulate.py  --  Synthetic fluorescence image simulator.

Models Yb-171 atom array on sCMOS camera:
  - Poisson photon statistics
  - Gaussian readout noise
  - PSF Gaussian profile per site
  - Background photons
  - PSF tail crosstalk from neighbors
  - Per-frame atom survival (stochastic loss)
"""

import numpy as np
from numpy.typing import NDArray
from src.config import SystemConfig
from src.psf import make_gaussian_template


def _frame_size(cfg: SystemConfig) -> tuple[int, int]:
    pitch = cfg.array.site_pitch_px
    margin = pitch
    H = cfg.array.n_rows * pitch + 2 * margin
    W = cfg.array.n_cols * pitch + 2 * margin
    return H, W


def simulate_frame(atom_states: NDArray[np.bool_],
                   cfg: SystemConfig,
                   rng: np.random.Generator,
                   exposure_ms: float,
                   roi_offsets: NDArray[np.float32] | None = None
                   ) -> NDArray[np.float32]:
    """
    Render a single fluorescence frame.

    Parameters
    ----------
    atom_states : (K,) bool -- True = atom present at site k
    cfg         : SystemConfig
    rng         : numpy random generator
    exposure_ms : exposure time in ms
    roi_offsets : (K,2) sub-pixel drift offsets (ignored for full-frame sim)

    Returns
    -------
    frame : (H, W) float32 -- photoelectron image
    """
    H, W = _frame_size(cfg)
    frame = np.zeros((H, W), dtype=np.float32)

    pitch  = cfg.array.site_pitch_px
    margin = pitch
    sigma  = cfg.psf.sigma_px
    R      = cfg.array.roi_size
    exp_s  = exposure_ms * 1e-3

    n_photons_mean = (cfg.physics.photon_rate
                      * cfg.physics.qe
                      * exp_s)               # mean photons per atom
    n_bg_mean      = (cfg.physics.bg_photon_rate
                      * cfg.physics.qe
                      * exp_s)               # mean bg photons per site

    K = cfg.array.n_sites

    for k in range(K):
        row = k // cfg.array.n_cols
        col = k %  cfg.array.n_cols
        cr  = row * pitch + margin   # center row in full frame
        cc  = col * pitch + margin   # center col in full frame

        # Background
        bg_counts = rng.poisson(n_bg_mean, size=(R, R)).astype(np.float32)
        r0 = cr - R // 2
        c0 = cc - R // 2
        frame[r0:r0+R, c0:c0+R] += bg_counts

        if atom_states[k]:
            # Signal: PSF-distributed Poisson photons
            tmpl = make_gaussian_template(R, sigma)   # (R,R), unit norm
            # Expected signal image: photon counts ~ N * tmpl (unnormalized)
            # Un-normalize to get raw pixel expectation
            tmpl_raw = (tmpl / (tmpl.max() + 1e-12)) * n_photons_mean
            signal   = rng.poisson(tmpl_raw.clip(0)).astype(np.float32)
            frame[r0:r0+R, c0:c0+R] += signal

    # Crosstalk: add fraction of neighbor signal
    ct = cfg.physics.crosstalk_fraction
    if ct > 0:
        # Simple approximation: shift copies of the frame
        for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
            shifted = np.roll(frame, (dr*pitch, dc*pitch), axis=(0,1))
            frame += ct * shifted

    # Readout noise (Gaussian, independent per pixel)
    read_noise = rng.normal(0.0, cfg.physics.readout_noise_e,
                             size=frame.shape).astype(np.float32)
    frame += read_noise
    return frame.clip(0)


def simulate_atom_states(K: int, p_survival: float,
                         prev_states: NDArray[np.bool_] | None,
                         rng: np.random.Generator
                         ) -> NDArray[np.bool_]:
    """
    Propagate atom states by one frame with stochastic loss.
    Initial call: prev_states=None -> random 50% loading.
    """
    if prev_states is None:
        return rng.random(K) < 0.60   # ~60% loading probability
    survive = rng.random(K) < p_survival
    return prev_states & survive


def build_site_centers(cfg: SystemConfig) -> NDArray[np.int32]:
    pitch  = cfg.array.site_pitch_px
    margin = pitch
    rows   = np.arange(cfg.array.n_rows)
    cols   = np.arange(cfg.array.n_cols)
    rr, cc = np.meshgrid(rows, cols, indexing="ij")
    return np.stack([rr.ravel() * pitch + margin,
                     cc.ravel() * pitch + margin], axis=1).astype(np.int32)
'''

# ── eval/benchmark.py ──────────────────────────────────────────────────────────
benchmark_py = '''"""
benchmark.py  --  Evaluation harness for MF-Array two-layer system.

Metrics:
  FAR          False Alarm Rate  = FP / (FP + TN)
  MDR          Miss Detection Rate = FN / (FN + TP)
  Fidelity     = 1 - (FAR + MDR) / 2   (or exact: TP_rate * P(H1) + TN_rate * P(H0))
  L2 routing rate = sites routed to L2 / total decisions
  Erasure rate = ERASURE_LOSS triggers / total decisions
  Throughput   = frames processed per second (wall clock)
"""

import sys, time, json, pathlib
import numpy as np

sys.path.insert(0, str(pathlib.Path(__file__).resolve().parents[1]))

from src.config import SystemConfig, MFConfig
from src.psf import build_template_array
from src.mf_detector import (compute_mf_scores, scores_to_probs,
                               extract_rois, calibrate_mf_params)
from src.simulate import (simulate_frame, simulate_atom_states,
                           build_site_centers)
from src.layer1_mf import Decision
from src.system import MFSystem


# ─────────────────────────────────────────────────────────────────────────────
def calibrate_system(cfg: SystemConfig,
                     n_calib: int = 50_000,
                     rng: np.random.Generator | None = None
                     ) -> tuple[float, float, float, float]:
    """
    Estimate MF score distributions from simulated calibration data.
    Returns (mu_atom, sigma_atom, mu_bg, sigma_bg).
    """
    if rng is None:
        rng = np.random.default_rng(42)

    K   = cfg.array.n_sites
    R   = cfg.array.roi_size
    templates = build_template_array(cfg.array, cfg.psf)
    tmpl = templates.mean(axis=0)
    tmpl /= (np.linalg.norm(tmpl) + 1e-12)
    tmpl_rep = tmpl[np.newaxis]

    rois_atom_list, rois_bg_list = [], []

    n_frames = n_calib // K + 1
    for _ in range(n_frames):
        # half atom, half empty for calibration
        states = np.zeros(K, dtype=bool)
        states[:K//2] = True
        rng.shuffle(states)

        frame = simulate_frame(states, cfg, rng, cfg.l1.exposure_ms)
        centers = build_site_centers(cfg)
        rois = extract_rois(frame, centers, R)

        rois_atom_list.append(rois[states])
        rois_bg_list.append(rois[~states])

    rois_atom = np.concatenate(rois_atom_list, axis=0)
    rois_bg   = np.concatenate(rois_bg_list,   axis=0)

    t_a = np.broadcast_to(tmpl_rep, (rois_atom.shape[0],) + tmpl.shape)
    t_b = np.broadcast_to(tmpl_rep, (rois_bg.shape[0],)   + tmpl.shape)
    s_a = compute_mf_scores(rois_atom, t_a)
    s_b = compute_mf_scores(rois_bg,   t_b)

    return (float(s_a.mean()), max(float(s_a.std()), 1e-6),
            float(s_b.mean()), max(float(s_b.std()), 1e-6))


# ─────────────────────────────────────────────────────────────────────────────
def run_benchmark(cfg: SystemConfig,
                  n_frames: int = 500,
                  seed: int = 0) -> dict:
    """
    Full benchmark: calibrate, run N frames, compute all metrics.
    """
    rng = np.random.default_rng(seed)
    print(f"Calibrating MF parameters on {cfg.array.n_sites * 200} samples ...")
    mu_atom, sig_atom, mu_bg, sig_bg = calibrate_system(cfg, n_calib=200*cfg.array.n_sites, rng=rng)
    print(f"  mu_atom={mu_atom:.3f}, sigma_atom={sig_atom:.3f}")
    print(f"  mu_bg  ={mu_bg:.3f},   sigma_bg  ={sig_bg:.3f}")
    print(f"  SNR    ={(mu_atom-mu_bg)/((sig_atom+sig_bg)/2):.2f}")

    system = MFSystem(cfg)
    system.set_mf_params(mu_atom, sig_atom, mu_bg, sig_bg)

    K     = cfg.array.n_sites
    # accumulators
    tp = fp = tn = fn = 0
    n_l2_total = n_erasure_total = n_decisions_total = 0
    atom_states = None

    t0 = time.perf_counter()
    for frame_idx in range(n_frames):
        atom_states = simulate_atom_states(K, cfg.physics.p_survival,
                                           atom_states, rng)
        frame_l1 = simulate_frame(atom_states, cfg, rng, cfg.l1.exposure_ms)
        frame_l2 = simulate_frame(atom_states, cfg, rng, cfg.l2.exposure_ms)

        result = system.process_frame(frame_l1, frame_l2)

        n_l2_total        += result.n_l2_routed
        n_erasure_total   += result.n_erasure
        n_decisions_total += K

        # Evaluate only ATOM_PRESENT / ATOM_ABSENT decisions (skip ERASURE for FAR/MDR)
        final_dec = result.decisions.copy()
        # Treat ERASURE as ATOM_ABSENT for this evaluation
        erasure_mask = final_dec == int(Decision.ERASURE_LOSS)
        final_dec[erasure_mask] = int(Decision.ATOM_ABSENT)

        pred_1 = final_dec == int(Decision.ATOM_PRESENT)
        true_1 = atom_states

        tp += int(( pred_1 &  true_1).sum())
        fp += int(( pred_1 & ~true_1).sum())
        tn += int((~pred_1 & ~true_1).sum())
        fn += int((~pred_1 &  true_1).sum())

    elapsed = time.perf_counter() - t0

    far = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    mdr = fn / (fn + tp) if (fn + tp) > 0 else 0.0
    fidelity = 1.0 - (far + mdr) / 2.0

    total_positive = tp + fn
    total_negative = fp + tn

    results = {
        "n_frames"       : n_frames,
        "n_sites"        : K,
        "total_decisions": n_decisions_total,
        "TP": tp, "FP": fp, "TN": tn, "FN": fn,
        "FAR"            : far,
        "MDR"            : mdr,
        "Fidelity"       : fidelity,
        "L2_routing_rate": n_l2_total  / n_decisions_total,
        "erasure_rate"   : n_erasure_total / n_decisions_total,
        "fps"            : n_frames / elapsed,
        "ms_per_frame"   : elapsed / n_frames * 1000,
        "mf_mu_atom"     : mu_atom,
        "mf_sig_atom"    : sig_atom,
        "mf_mu_bg"       : mu_bg,
        "mf_sig_bg"      : sig_bg,
        "mf_snr"         : (mu_atom - mu_bg) / ((sig_atom + sig_bg) / 2),
        "l1_theta_H"     : cfg.l1.theta_H,
        "l1_theta_L"     : cfg.l1.theta_L,
        "l1_exposure_ms" : cfg.l1.exposure_ms,
        "l2_exposure_ms" : cfg.l2.exposure_ms,
    }
    return results


# ─────────────────────────────────────────────────────────────────────────────
def print_report(r: dict) -> None:
    print("\\n" + "="*52)
    print("  MF-Array Two-Layer System — Benchmark Report")
    print("="*52)
    print(f"  Frames : {r[\'n_frames\']:,}   Sites : {r[\'n_sites\']}   "
          f"Total decisions : {r[\'total_decisions\']:,}")
    print(f"  TP={r[\'TP\']:,}  FP={r[\'FP\']:,}  TN={r[\'TN\']:,}  FN={r[\'FN\']:,}")
    print(f"  FAR          : {r[\'FAR\']:.6f}  ({r[\'FAR\']*100:.4f} %)")
    print(f"  MDR          : {r[\'MDR\']:.6f}  ({r[\'MDR\']*100:.4f} %)")
    print(f"  Fidelity     : {r[\'Fidelity\']:.6f}  ({r[\'Fidelity\']*100:.4f} %)")
    print(f"  L2 routing   : {r[\'L2_routing_rate\']*100:.3f} %")
    print(f"  Erasure rate : {r[\'erasure_rate\']*100:.4f} %")
    print(f"  MF SNR       : {r[\'mf_snr\']:.2f}")
    print(f"  Speed        : {r[\'fps\']:.1f} frame/s  "
          f"({r[\'ms_per_frame\']:.1f} ms/frame)")
    print("="*52)


# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    cfg = SystemConfig()
    results = run_benchmark(cfg, n_frames=500, seed=42)
    print_report(results)

    out_path = pathlib.Path(__file__).parent.parent / "results" / "benchmark.json"
    out_path.parent.mkdir(exist_ok=True)
    with open(out_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"\\nResults saved to {out_path}")
'''

# write files
files = {
    "src/mf_detector.py" : mf_detector,
    "src/layer1_mf.py"   : layer1_mf,
    "src/layer2_mf.py"   : layer2_mf,
    "src/system.py"      : system_py,
    "src/simulate.py"    : simulate_py,
    "eval/benchmark.py"  : benchmark_py,
}

for rel, content in files.items():
    path = f"{base}/{rel}"
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as fh:
        fh.write(content)
    print(f"wrote {rel}  ({len(content)} chars)")

# __init__ files
for d in ["src", "eval"]:
    init = f"{base}/{d}/__init__.py"
    with open(init, "w") as fh:
        fh.write("")
    
print("Done.")


wrote src/mf_detector.py  (4894 chars)
wrote src/layer1_mf.py  (4121 chars)
wrote src/layer2_mf.py  (976 chars)
wrote src/system.py  (5485 chars)
wrote src/simulate.py  (4266 chars)
wrote eval/benchmark.py  (7266 chars)
Done.


In [7]:

base = "/data/workspace/61448ef7-eb83-45fe-a53b-e87021227cc1/mf_atom_detection"

readme = '''# mf-atom-detection

**PA-3HDA-MF v1.0** — Matched Filter Array two-layer atom detection system for neutral atom quantum computing.

## Architecture

Replaces L1 / L2 lightweight CNNs from PA-3HDA v10.0 with a **Matched Filter (MF) Array**.
L1.5 physical-diagnostic router is removed. Architecture reduced to two active layers:

```
Frame (short exp.)  ──► L1: MF-Array + Bayesian fusion + CUSUM ──► ATOM_PRESENT
                                                                 ──► ATOM_ABSENT
                                                                 ──► ERASURE_LOSS
                                                                 ──► ROUTE_L2
Frame (long  exp.)  ──► L2: MF-Array precise judgment ──────────► ATOM_PRESENT / ATOM_ABSENT
```

### Why Matched Filter?

The Matched Filter is the SNR-optimal linear detector for a known signal in additive white
Gaussian noise (Neyman-Pearson lemma). For a Gaussian PSF atom signal:

```
score_k = dot(roi_k.flat, h_k.flat)          # h_k: unit-norm Gaussian template
p_mf(k) = sigmoid( LLR(score_k) )            # calibrated via Bayesian LRT
```

Advantages over CNN:
- Zero learned parameters — no training required
- Theoretically optimal for the physical noise model
- Pure dot-product: faster inference, FPGA-friendly
- Graceful degradation as SNR drops (analytical performance bound)

### Layer Details

| Layer | Method | Exposure | Notes |
|-------|--------|----------|-------|
| L1 | MF Array + Bayesian log-odds temporal fusion + CUSUM | t_L1 (short) | Covers ≥99.5% of sites; ERASURE output for atom loss |
| L2 | MF Array precise judgment | t_L2 (long) | Final adjudication for uncertain sites (<0.5%) |

### Five-Type Structured Output

| # | Output | Source |
|---|--------|--------|
| 1 | `ATOM_PRESENT`  | L1 / L2 |
| 2 | `ATOM_ABSENT`   | L1 / L2 |
| 3 | `ERASURE_LOSS`  | L1 CUSUM |
| 4 | `CORR_LOSS`     | (interface reserved, L1.5 removed) |
| 5 | `DRIFT_ALARM`   | L0 background (stubbed) |

## Repository Structure

```
mf_atom_detection/
├── src/
│   ├── config.py        System-wide parameters
│   ├── psf.py           Gaussian PSF model, template generation
│   ├── mf_detector.py   Vectorized MF score computation
│   ├── layer1_mf.py     L1: MF + Bayesian fusion + CUSUM
│   ├── layer2_mf.py     L2: MF precise judgment
│   ├── simulate.py      Synthetic fluorescence image simulator
│   └── system.py        Two-layer system orchestration
├── eval/
│   └── benchmark.py     FAR / MDR / Fidelity / routing rate / speed
├── results/
│   └── benchmark.json   Latest evaluation results
└── README.md
```

## Quick Start

```bash
pip install numpy
python eval/benchmark.py
```

## Evaluation Results

See `results/benchmark.json` for the latest run.

Key metrics (default config: 20×20 array, 400 sites, 500 frames):

| Metric | Value |
|--------|-------|
| Fidelity | see results/benchmark.json |
| FAR | see results/benchmark.json |
| MDR | see results/benchmark.json |
| L2 routing rate | see results/benchmark.json |
| Speed | see results/benchmark.json |

## Physical Parameters (default)

| Parameter | Value | Source |
|-----------|-------|--------|
| Photon rate | 35,000 ph/s/atom | 171Yb, 399 nm |
| Camera QE | 72% | Qbit 4610 sCMOS |
| Readout noise | 0.3 e⁻ | sCMOS spec |
| PSF sigma | 4.6 px | NA=0.5 objective |
| Site pitch | 54 px | 5 µm spacing |
| t_L1 | 5.09 ms | ~128 photons/atom |
| t_L2 | 18.0 ms | independently calibrated |
| p_survival | 0.9952 | 0.48%/frame loss |

## Reference

PA-3HDA v10.0 patent disclosure — Hierarchical Adaptive Fluorescence Detection System
and Method for Neutral Atom Arrays in Quantum Computing (2026).
'''

req = "numpy>=1.24\n"

with open(f"{base}/README.md", "w") as f:
    f.write(readme)
with open(f"{base}/requirements.txt", "w") as f:
    f.write(req)

print("README.md and requirements.txt written.")


README.md and requirements.txt written.


In [11]:

import sys, json, time, pathlib
import numpy as np

base = "/data/workspace/61448ef7-eb83-45fe-a53b-e87021227cc1/mf_atom_detection"
sys.path.insert(0, base)

from src.config import SystemConfig
from src.psf import build_template_array
from src.mf_detector import compute_mf_scores, scores_to_probs, extract_rois
from src.simulate import simulate_frame, simulate_atom_states, build_site_centers
from src.layer1_mf import Decision
from src.system import MFSystem

# ── calibration ──────────────────────────────────────────────────────────────
def calibrate_system(cfg, n_calib=20000, rng=None):
    if rng is None: rng = np.random.default_rng(42)
    K = cfg.array.n_sites; R = cfg.array.roi_size
    templates = build_template_array(cfg.array, cfg.psf)
    tmpl = templates.mean(axis=0); tmpl /= (np.linalg.norm(tmpl)+1e-12)
    tmpl_rep = tmpl[np.newaxis]
    rois_atom_list, rois_bg_list = [], []
    n_frames = max(n_calib // K, 10)
    for _ in range(n_frames):
        states = np.zeros(K, dtype=bool); states[:K//2]=True; rng.shuffle(states)
        frame = simulate_frame(states, cfg, rng, cfg.l1.exposure_ms)
        centers = build_site_centers(cfg)
        rois = extract_rois(frame, centers, R)
        rois_atom_list.append(rois[states]); rois_bg_list.append(rois[~states])
    rois_atom = np.concatenate(rois_atom_list); rois_bg = np.concatenate(rois_bg_list)
    ta = np.broadcast_to(tmpl_rep, (rois_atom.shape[0],)+tmpl.shape)
    tb = np.broadcast_to(tmpl_rep, (rois_bg.shape[0],)  +tmpl.shape)
    sa = compute_mf_scores(rois_atom, ta); sb = compute_mf_scores(rois_bg, tb)
    return float(sa.mean()), max(float(sa.std()),1e-6), float(sb.mean()), max(float(sb.std()),1e-6)

# ── benchmark ─────────────────────────────────────────────────────────────────
def run_benchmark(cfg, n_frames=500, seed=0):
    rng = np.random.default_rng(seed)
    print(f"Calibrating MF params on {cfg.array.n_sites*50} samples …")
    mu_atom,sig_atom,mu_bg,sig_bg = calibrate_system(cfg, n_calib=cfg.array.n_sites*50, rng=np.random.default_rng(99))
    snr = (mu_atom-mu_bg)/((sig_atom+sig_bg)/2)
    print(f"  mu_atom={mu_atom:.3f} ± {sig_atom:.3f}  |  mu_bg={mu_bg:.3f} ± {sig_bg:.3f}  |  SNR={snr:.2f}")

    system = MFSystem(cfg); system.set_mf_params(mu_atom,sig_atom,mu_bg,sig_bg)
    K = cfg.array.n_sites
    tp=fp=tn=fn=0; n_l2=n_era=n_tot=0; atom_states=None
    t0 = time.perf_counter()
    for i in range(n_frames):
        atom_states = simulate_atom_states(K, cfg.physics.p_survival, atom_states, rng)
        fl1 = simulate_frame(atom_states, cfg, rng, cfg.l1.exposure_ms)
        fl2 = simulate_frame(atom_states, cfg, rng, cfg.l2.exposure_ms)
        res = system.process_frame(fl1, fl2)
        n_l2  += res.n_l2_routed; n_era += res.n_erasure; n_tot += K
        dec = res.decisions.copy()
        dec[dec==int(Decision.ERASURE_LOSS)] = int(Decision.ATOM_ABSENT)
        p1 = dec==int(Decision.ATOM_PRESENT); t1 = atom_states
        tp+=int(( p1& t1).sum()); fp+=int(( p1&~t1).sum())
        tn+=int((~p1&~t1).sum()); fn+=int((~p1& t1).sum())
    elapsed = time.perf_counter()-t0
    far = fp/(fp+tn) if (fp+tn)>0 else 0.0
    mdr = fn/(fn+tp) if (fn+tp)>0 else 0.0
    fid = 1.-(far+mdr)/2.
    return dict(n_frames=n_frames,n_sites=K,total_decisions=n_tot,
                TP=tp,FP=fp,TN=tn,FN=fn,FAR=far,MDR=mdr,Fidelity=fid,
                L2_routing_rate=n_l2/n_tot,erasure_rate=n_era/n_tot,
                fps=n_frames/elapsed, ms_per_frame=elapsed/n_frames*1000,
                mf_mu_atom=mu_atom,mf_sig_atom=sig_atom,mf_mu_bg=mu_bg,mf_sig_bg=sig_bg,
                mf_snr=snr,
                l1_theta_H=cfg.l1.theta_H,l1_theta_L=cfg.l1.theta_L,
                l1_exposure_ms=cfg.l1.exposure_ms,l2_exposure_ms=cfg.l2.exposure_ms)

cfg = SystemConfig()
print(cfg.summary())
r = run_benchmark(cfg, n_frames=500, seed=42)

# print report
print("\n" + "="*54)
print("  MF-Array Two-Layer System — Benchmark Report")
print("="*54)
print(f"  Array      : {r['n_sites']} sites  |  Frames: {r['n_frames']}  |  Decisions: {r['total_decisions']:,}")
print(f"  TP={r['TP']:,}  FP={r['FP']:,}  TN={r['TN']:,}  FN={r['FN']:,}")
print(f"  FAR        : {r['FAR']:.8f}  ({r['FAR']*100:.6f} %)")
print(f"  MDR        : {r['MDR']:.8f}  ({r['MDR']*100:.6f} %)")
print(f"  Fidelity   : {r['Fidelity']:.8f}  ({r['Fidelity']*100:.6f} %)")
print(f"  L2 routing : {r['L2_routing_rate']*100:.3f} %")
print(f"  Erasure    : {r['erasure_rate']*100:.4f} %")
print(f"  MF SNR     : {r['mf_snr']:.2f}")
print(f"  Speed      : {r['fps']:.1f} frame/s  ({r['ms_per_frame']:.2f} ms/frame)")
print("="*54)

# save results
out = pathlib.Path(base)/"results"/"benchmark.json"
out.parent.mkdir(exist_ok=True)
with open(out,"w") as f: json.dump(r, f, indent=2)
print(f"\nSaved → {out}")


=== MF-Array Two-Layer System Config ===
  Array      : 20x20 sites, pitch=54px, ROI=17px
  PSF sigma  : 4.6 px
  L1 exposure: 5.09 ms, theta=(0.92, 0.08)
  L2 exposure: 18.0 ms
  CUSUM h_loss: 5.0
  p_survival : 0.9952
Calibrating MF params on 20000 samples …


  mu_atom=1100.649 ± 13.992  |  mu_bg=63.807 ± 10.454  |  SNR=84.83


/data/workspace/61448ef7-eb83-45fe-a53b-e87021227cc1/mf_atom_detection/src/mf_detector.py:109: RuntimeWarning: overflow encountered in exp
  probs = 1.0 / (1.0 + np.exp(-llr.astype(np.float32)))


/data/workspace/61448ef7-eb83-45fe-a53b-e87021227cc1/mf_atom_detection/src/layer1_mf.py:105: RuntimeWarning: overflow encountered in exp
  p_post = 1.0 / (1.0 + np.exp(-state.log_odds)).astype(np.float32)



  MF-Array Two-Layer System — Benchmark Report
  Array      : 400 sites  |  Frames: 500  |  Decisions: 200,000
  TP=46,955  FP=0  TN=153,045  FN=0
  FAR        : 0.00000000  (0.000000 %)
  MDR        : 0.00000000  (0.000000 %)
  Fidelity   : 1.00000000  (100.000000 %)
  L2 routing : 0.000 %
  Erasure    : 76.5225 %
  MF SNR     : 84.83
  Speed      : 10.9 frame/s  (91.48 ms/frame)

Saved → /data/workspace/61448ef7-eb83-45fe-a53b-e87021227cc1/mf_atom_detection/results/benchmark.json


In [15]:

import sys, pathlib, json, time
import numpy as np

base = "/data/workspace/61448ef7-eb83-45fe-a53b-e87021227cc1/mf_atom_detection"
sys.path.insert(0, base)

# ── Fix 1: patch layer1_mf.py ─────────────────────────────────────────────────
# When ATOM_ABSENT is confidently decided, reset CUSUM (no atom to lose).
# Also clamp log-odds to avoid float32 overflow.

layer1_fixed = '''"""
layer1_mf.py  --  L1: Matched Filter fast-decision layer.

L1.5 (physical diagnostic router) is REMOVED.

Decision logic per site k at frame t:
  1. Compute MF score s(k,t) and convert to probability p_mf(k,t)
  2. Bayesian log-odds update:
       x(k,t) = x(k,t-1) + log(ps/(1-ps)) + lambda * log(p_mf/(1-p_mf))
       p_post  = sigmoid(x(k,t))
  3. CUSUM update  (only for ERASURE detection):
       S(k,t) = max(0, S(k,t-1) + log((1-p_mf)/p_mf))
     If S(k,t) > h_loss  -> ERASURE_LOSS  (reset S and mark site absent)
     If ATOM_ABSENT decided -> reset S to 0 (no atom -> nothing to lose)
  4. Double-threshold decision:
       p_post > theta_H  -> ATOM_PRESENT
       p_post < theta_L  -> ATOM_ABSENT
       otherwise         -> route to L2
"""

import numpy as np
from numpy.typing import NDArray
from enum import IntEnum

LOG_ODDS_CLIP = 20.0   # clip Bayesian log-odds to avoid float32 overflow


class Decision(IntEnum):
    ATOM_PRESENT  = 1
    ATOM_ABSENT   = 0
    ROUTE_L2      = -1
    ERASURE_LOSS  = 2
    CORR_LOSS     = 3
    DRIFT_ALARM   = 4


class L1State:
    def __init__(self, n_sites: int):
        self.n_sites = n_sites
        self.log_odds: NDArray[np.float32] = np.zeros(n_sites, dtype=np.float32)
        self.S_cusum:  NDArray[np.float32] = np.zeros(n_sites, dtype=np.float32)
        self.frame_count: int = 0

    def reset(self) -> None:
        self.log_odds[:] = 0.0
        self.S_cusum[:]  = 0.0
        self.frame_count  = 0


def _safe_logit(p: NDArray[np.float32], eps: float = 1e-6) -> NDArray[np.float32]:
    p = np.clip(p, eps, 1.0 - eps)
    return np.log(p / (1.0 - p))


def run_l1(p_mf:       NDArray[np.float32],
           state:      L1State,
           theta_H:    float,
           theta_L:    float,
           p_survival: float,
           lambda_obs: float,
           h_loss:     float,
           h_corr:     float,
           warmup:     int
           ) -> tuple[NDArray[np.int8], NDArray[np.float32]]:
    K = state.n_sites
    state.frame_count += 1
    use_temporal = state.frame_count > warmup

    # -- CUSUM update (loss detection)
    cusum_increment = _safe_logit(1.0 - p_mf)   # log((1-p)/p): positive when no atom
    cusum_increment = np.clip(cusum_increment, -LOG_ODDS_CLIP, LOG_ODDS_CLIP)
    state.S_cusum = np.maximum(0.0, state.S_cusum + cusum_increment)

    # -- Bayesian log-odds update
    survival_logit = float(np.log(p_survival / (1.0 - p_survival)))
    obs_logit = lambda_obs * _safe_logit(p_mf)

    if use_temporal:
        raw = state.log_odds + survival_logit + obs_logit
    else:
        raw = obs_logit

    state.log_odds = np.clip(raw, -LOG_ODDS_CLIP, LOG_ODDS_CLIP).astype(np.float32)
    p_post = 1.0 / (1.0 + np.exp(-state.log_odds)).astype(np.float32)
    p_post = np.clip(p_post, 1e-7, 1.0 - 1e-7).astype(np.float32)

    # -- Decision (priority: ERASURE > ACCEPT > REJECT > ROUTE_L2)
    decisions = np.full(K, int(Decision.ROUTE_L2), dtype=np.int8)

    # ERASURE: CUSUM threshold crossed
    erasure_mask = state.S_cusum > h_loss
    decisions[erasure_mask] = int(Decision.ERASURE_LOSS)
    state.S_cusum[erasure_mask] = 0.0   # reset after trigger

    # High-confidence accept / reject
    accept_mask = (~erasure_mask) & (p_post >= theta_H)
    reject_mask = (~erasure_mask) & (p_post <= theta_L)
    decisions[accept_mask] = int(Decision.ATOM_PRESENT)
    decisions[reject_mask] = int(Decision.ATOM_ABSENT)

    # Key fix: reset CUSUM for sites confidently decided ABSENT
    # (no atom present -> no loss event possible -> prevent false ERASURE)
    state.S_cusum[reject_mask] = 0.0

    return decisions, p_post
'''

with open(f"{base}/src/layer1_mf.py", "w") as f:
    f.write(layer1_fixed)
print("layer1_mf.py patched (CUSUM fix + log-odds clamp)")

# reload module
if "src.layer1_mf" in sys.modules:
    del sys.modules["src.layer1_mf"]
if "src.system" in sys.modules:
    del sys.modules["src.system"]
for k in list(sys.modules.keys()):
    if k.startswith("src."):
        del sys.modules[k]

from src.config import SystemConfig
from src.psf import build_template_array
from src.mf_detector import compute_mf_scores, scores_to_probs, extract_rois
from src.simulate import simulate_frame, simulate_atom_states, build_site_centers
from src.layer1_mf import Decision
from src.system import MFSystem

# ── re-run benchmark ──────────────────────────────────────────────────────────
def calibrate_system(cfg, n_calib=20000, rng=None):
    if rng is None: rng = np.random.default_rng(42)
    K = cfg.array.n_sites; R = cfg.array.roi_size
    templates = build_template_array(cfg.array, cfg.psf)
    tmpl = templates.mean(axis=0); tmpl /= (np.linalg.norm(tmpl)+1e-12)
    tmpl_rep = tmpl[np.newaxis]
    ra, rb = [], []
    for _ in range(max(n_calib//K, 10)):
        states = np.zeros(K, dtype=bool); states[:K//2]=True; rng.shuffle(states)
        frame = simulate_frame(states, cfg, rng, cfg.l1.exposure_ms)
        rois  = extract_rois(frame, build_site_centers(cfg), R)
        ra.append(rois[states]); rb.append(rois[~states])
    sa_arr = np.concatenate(ra); sb_arr = np.concatenate(rb)
    ta = np.broadcast_to(tmpl_rep,(sa_arr.shape[0],)+tmpl.shape)
    tb = np.broadcast_to(tmpl_rep,(sb_arr.shape[0],)+tmpl.shape)
    sa = compute_mf_scores(sa_arr,ta); sb = compute_mf_scores(sb_arr,tb)
    return float(sa.mean()),max(float(sa.std()),1e-6),float(sb.mean()),max(float(sb.std()),1e-6)

def run_benchmark(cfg, n_frames=500, seed=0):
    rng = np.random.default_rng(seed)
    print(f"Calibrating ({cfg.array.n_sites*50} samples) …")
    mu_a,sg_a,mu_b,sg_b = calibrate_system(cfg, cfg.array.n_sites*50, np.random.default_rng(99))
    snr = (mu_a-mu_b)/((sg_a+sg_b)/2)
    print(f"  mu_atom={mu_a:.2f}±{sg_a:.2f}  mu_bg={mu_b:.2f}±{sg_b:.2f}  SNR={snr:.1f}")
    system = MFSystem(cfg); system.set_mf_params(mu_a,sg_a,mu_b,sg_b)
    K = cfg.array.n_sites
    tp=fp=tn=fn=0; n_l2=n_era=n_tot=0; atom_states=None
    t0=time.perf_counter()
    for _ in range(n_frames):
        atom_states = simulate_atom_states(K,cfg.physics.p_survival,atom_states,rng)
        fl1 = simulate_frame(atom_states,cfg,rng,cfg.l1.exposure_ms)
        fl2 = simulate_frame(atom_states,cfg,rng,cfg.l2.exposure_ms)
        res = system.process_frame(fl1,fl2)
        n_l2+=res.n_l2_routed; n_era+=res.n_erasure; n_tot+=K
        dec=res.decisions.copy()
        dec[dec==int(Decision.ERASURE_LOSS)]=int(Decision.ATOM_ABSENT)
        p1=dec==int(Decision.ATOM_PRESENT); t1=atom_states
        tp+=int((p1&t1).sum()); fp+=int((p1&~t1).sum())
        tn+=int((~p1&~t1).sum()); fn+=int((~p1&t1).sum())
    elapsed=time.perf_counter()-t0
    far=fp/(fp+tn) if (fp+tn)>0 else 0.0
    mdr=fn/(fn+tp) if (fn+tp)>0 else 0.0
    return dict(n_frames=n_frames,n_sites=K,total_decisions=n_tot,
                TP=tp,FP=fp,TN=tn,FN=fn,FAR=far,MDR=mdr,Fidelity=1.-(far+mdr)/2.,
                L2_routing_rate=n_l2/n_tot,erasure_rate=n_era/n_tot,
                fps=n_frames/elapsed,ms_per_frame=elapsed/n_frames*1000,
                mf_mu_atom=mu_a,mf_sig_atom=sg_a,mf_mu_bg=mu_b,mf_sig_bg=sg_b,mf_snr=snr,
                l1_theta_H=cfg.l1.theta_H,l1_theta_L=cfg.l1.theta_L,
                l1_exposure_ms=cfg.l1.exposure_ms,l2_exposure_ms=cfg.l2.exposure_ms)

cfg = SystemConfig()
r = run_benchmark(cfg, n_frames=500, seed=42)

print("\n" + "="*56)
print("  MF-Array Two-Layer System — Benchmark Report  (v1.1)")
print("="*56)
print(f"  Array      : {r['n_sites']} sites  |  Frames: {r['n_frames']}  |  Decisions: {r['total_decisions']:,}")
print(f"  TP={r['TP']:,}  FP={r['FP']:,}  TN={r['TN']:,}  FN={r['FN']:,}")
print(f"  FAR        : {r['FAR']:.8f}   ({r['FAR']*1e6:.2f} ppm)")
print(f"  MDR        : {r['MDR']:.8f}   ({r['MDR']*1e6:.2f} ppm)")
print(f"  Fidelity   : {r['Fidelity']:.8f}   ({r['Fidelity']*100:.6f} %)")
print(f"  L2 routing : {r['L2_routing_rate']*100:.4f} %")
print(f"  Erasure    : {r['erasure_rate']*100:.4f} %")
print(f"  MF SNR     : {r['mf_snr']:.1f}")
print(f"  Speed      : {r['fps']:.1f} fps  ({r['ms_per_frame']:.1f} ms/frame)")
print("="*56)

out = pathlib.Path(base)/"results"/"benchmark.json"
with open(out,"w") as f: json.dump(r,f,indent=2)
print(f"\nSaved → {out}")


layer1_mf.py patched (CUSUM fix + log-odds clamp)
Calibrating (20000 samples) …


  mu_atom=1100.65±13.99  mu_bg=63.81±10.45  SNR=84.8


/data/workspace/61448ef7-eb83-45fe-a53b-e87021227cc1/mf_atom_detection/src/mf_detector.py:109: RuntimeWarning: overflow encountered in exp
  probs = 1.0 / (1.0 + np.exp(-llr.astype(np.float32)))



  MF-Array Two-Layer System — Benchmark Report  (v1.1)
  Array      : 400 sites  |  Frames: 500  |  Decisions: 200,000
  TP=46,955  FP=0  TN=153,045  FN=0
  FAR        : 0.00000000   (0.00 ppm)
  MDR        : 0.00000000   (0.00 ppm)
  Fidelity   : 1.00000000   (100.000000 %)
  L2 routing : 0.0000 %
  Erasure    : 76.5225 %
  MF SNR     : 84.8
  Speed      : 10.8 fps  (92.6 ms/frame)

Saved → /data/workspace/61448ef7-eb83-45fe-a53b-e87021227cc1/mf_atom_detection/results/benchmark.json


In [19]:

import sys, pathlib, json, time
import numpy as np
import warnings; warnings.filterwarnings("ignore")

base = "/data/workspace/61448ef7-eb83-45fe-a53b-e87021227cc1/mf_atom_detection"
sys.path.insert(0, base)

# ── rewrite layer1_mf.py with proper CUSUM gating ────────────────────────────
layer1_v2 = r'''"""
layer1_mf.py  --  L1: Matched Filter fast-decision layer (PA-3HDA-MF v1.0).

Key design: CUSUM is gated on *previous-frame* Bayesian belief.
  - Only accumulates loss evidence when log_odds_{t-1} > 0 (atom believed present).
  - Prevents false ERASURE for perpetually empty sites.
  - L1.5 router removed; direct L2 routing for uncertain sites.
"""

import numpy as np
from numpy.typing import NDArray
from enum import IntEnum

LOG_ODDS_CLIP = 15.0


class Decision(IntEnum):
    ATOM_PRESENT = 1
    ATOM_ABSENT  = 0
    ROUTE_L2     = -1
    ERASURE_LOSS = 2
    CORR_LOSS    = 3
    DRIFT_ALARM  = 4


class L1State:
    def __init__(self, n_sites: int):
        self.n_sites = n_sites
        self.log_odds   = np.zeros(n_sites, dtype=np.float64)
        self.S_cusum    = np.zeros(n_sites, dtype=np.float64)
        self.frame_count: int = 0

    def reset(self) -> None:
        self.log_odds[:] = 0.0
        self.S_cusum[:]  = 0.0
        self.frame_count  = 0


def _safe_logit64(p: NDArray, eps: float = 1e-9) -> NDArray:
    p = np.clip(p, eps, 1.0 - eps)
    return np.log(p / (1.0 - p))


def run_l1(p_mf, state, theta_H, theta_L,
           p_survival, lambda_obs, h_loss, h_corr, warmup):
    """
    Returns (decisions: int8 K, p_post: float32 K).
    """
    K = state.n_sites
    state.frame_count += 1
    use_temporal = state.frame_count > warmup

    p_mf = p_mf.astype(np.float64)

    # -- Save previous belief for CUSUM gating
    prev_log_odds = state.log_odds.copy()

    # -- Bayesian log-odds update
    survival_logit = np.log(p_survival / (1.0 - p_survival))
    obs_logit      = lambda_obs * _safe_logit64(p_mf)

    if use_temporal:
        raw = state.log_odds + survival_logit + obs_logit
    else:
        raw = obs_logit  # warmup: single-frame fallback

    state.log_odds = np.clip(raw, -LOG_ODDS_CLIP, LOG_ODDS_CLIP)

    p_post = (1.0 / (1.0 + np.exp(-state.log_odds))).astype(np.float32)
    p_post = np.clip(p_post, 1e-7, 1.0 - 1e-7)

    # -- CUSUM update (gated: only when prev belief > 0)
    cusum_incr = _safe_logit64(1.0 - p_mf)          # log((1-p)/p) > 0 when no atom
    cusum_incr = np.clip(cusum_incr, -LOG_ODDS_CLIP, LOG_ODDS_CLIP)

    # Gate: accumulate only when previous frame believed atom was present
    gate = (prev_log_odds > 0.0).astype(np.float64)
    state.S_cusum = np.maximum(0.0, state.S_cusum + gate * cusum_incr)

    # -- Decision (priority: ERASURE > ACCEPT > REJECT > ROUTE_L2)
    decisions = np.full(K, int(Decision.ROUTE_L2), dtype=np.int8)

    erasure_mask = state.S_cusum > h_loss
    decisions[erasure_mask] = int(Decision.ERASURE_LOSS)
    state.S_cusum[erasure_mask] = 0.0       # reset after trigger

    accept_mask = (~erasure_mask) & (p_post >= theta_H)
    reject_mask = (~erasure_mask) & (p_post <= theta_L)
    decisions[accept_mask] = int(Decision.ATOM_PRESENT)
    decisions[reject_mask] = int(Decision.ATOM_ABSENT)
    # Reset CUSUM for confidently absent sites
    state.S_cusum[reject_mask] = 0.0

    return decisions, p_post
'''

with open(f"{base}/src/layer1_mf.py", "w") as f:
    f.write(layer1_v2)
print("layer1_mf.py v2 written (prev-frame CUSUM gating)")

# also fix overflow in mf_detector.py
mf_fix = open(f"{base}/src/mf_detector.py").read()
mf_fix = mf_fix.replace(
    "probs = 1.0 / (1.0 + np.exp(-llr.astype(np.float32)))",
    "llr_clipped = np.clip(llr, -20.0, 20.0)\n    probs = (1.0 / (1.0 + np.exp(-llr_clipped))).astype(np.float32)"
)
with open(f"{base}/src/mf_detector.py", "w") as f:
    f.write(mf_fix)
print("mf_detector.py overflow fix applied")

# reload all src modules
for k in list(sys.modules.keys()):
    if k.startswith("src."):
        del sys.modules[k]

from src.config import SystemConfig
from src.psf import build_template_array
from src.mf_detector import compute_mf_scores, scores_to_probs, extract_rois
from src.simulate import simulate_frame, simulate_atom_states, build_site_centers
from src.layer1_mf import Decision
from src.system import MFSystem

# ── calibration ───────────────────────────────────────────────────────────────
def calibrate_system(cfg, n_calib=20000, rng=None):
    if rng is None: rng = np.random.default_rng(42)
    K = cfg.array.n_sites; R = cfg.array.roi_size
    tmpl = build_template_array(cfg.array, cfg.psf).mean(axis=0)
    tmpl /= np.linalg.norm(tmpl) + 1e-12
    tmpl1 = tmpl[np.newaxis]
    ra, rb = [], []
    for _ in range(max(n_calib // K, 10)):
        states = np.zeros(K, dtype=bool); states[:K//2] = True; rng.shuffle(states)
        frame = simulate_frame(states, cfg, rng, cfg.l1.exposure_ms)
        rois  = extract_rois(frame, build_site_centers(cfg), R)
        ra.append(rois[states]); rb.append(rois[~states])
    sa_arr = np.concatenate(ra); sb_arr = np.concatenate(rb)
    ta = np.broadcast_to(tmpl1, (sa_arr.shape[0],) + tmpl.shape)
    tb = np.broadcast_to(tmpl1, (sb_arr.shape[0],) + tmpl.shape)
    sa = compute_mf_scores(sa_arr, ta); sb = compute_mf_scores(sb_arr, tb)
    return (float(sa.mean()), max(float(sa.std()), 1e-6),
            float(sb.mean()), max(float(sb.std()), 1e-6))

# ── benchmark ─────────────────────────────────────────────────────────────────
def run_benchmark(cfg, n_frames=500, seed=0):
    rng = np.random.default_rng(seed)
    print("Calibrating …", end=" ", flush=True)
    mu_a,sg_a,mu_b,sg_b = calibrate_system(cfg, cfg.array.n_sites*50, np.random.default_rng(99))
    snr = (mu_a - mu_b) / ((sg_a + sg_b) / 2)
    print(f"mu_atom={mu_a:.1f}±{sg_a:.1f}  mu_bg={mu_b:.1f}±{sg_b:.1f}  SNR={snr:.1f}")

    system = MFSystem(cfg); system.set_mf_params(mu_a, sg_a, mu_b, sg_b)
    K = cfg.array.n_sites
    tp=fp=tn=fn=0; n_l2=n_era=n_tot=0; atom_states=None
    t0 = time.perf_counter()
    for _ in range(n_frames):
        atom_states = simulate_atom_states(K, cfg.physics.p_survival, atom_states, rng)
        fl1 = simulate_frame(atom_states, cfg, rng, cfg.l1.exposure_ms)
        fl2 = simulate_frame(atom_states, cfg, rng, cfg.l2.exposure_ms)
        res = system.process_frame(fl1, fl2)
        n_l2 += res.n_l2_routed; n_era += res.n_erasure; n_tot += K
        dec = res.decisions.copy()
        dec[dec == int(Decision.ERASURE_LOSS)] = int(Decision.ATOM_ABSENT)
        p1 = dec == int(Decision.ATOM_PRESENT); t1 = atom_states
        tp += int(( p1 &  t1).sum()); fp += int(( p1 & ~t1).sum())
        tn += int((~p1 & ~t1).sum()); fn += int((~p1 &  t1).sum())
    elapsed = time.perf_counter() - t0
    far = fp/(fp+tn) if (fp+tn)>0 else 0.0
    mdr = fn/(fn+tp) if (fn+tp)>0 else 0.0
    return dict(n_frames=n_frames, n_sites=K, total_decisions=n_tot,
                TP=tp, FP=fp, TN=tn, FN=fn,
                FAR=far, MDR=mdr, Fidelity=1.-(far+mdr)/2.,
                L2_routing_rate=n_l2/n_tot, erasure_rate=n_era/n_tot,
                fps=n_frames/elapsed, ms_per_frame=elapsed/n_frames*1000,
                mf_mu_atom=mu_a, mf_sig_atom=sg_a, mf_mu_bg=mu_b, mf_sig_bg=sg_b,
                mf_snr=snr, l1_theta_H=cfg.l1.theta_H, l1_theta_L=cfg.l1.theta_L,
                l1_exposure_ms=cfg.l1.exposure_ms, l2_exposure_ms=cfg.l2.exposure_ms)

cfg = SystemConfig()
r = run_benchmark(cfg, n_frames=500, seed=42)

print("\n" + "="*56)
print("  MF-Array Two-Layer System — Benchmark (PA-3HDA-MF v1.0)")
print("="*56)
print(f"  Sites={r['n_sites']}  Frames={r['n_frames']}  Decisions={r['total_decisions']:,}")
print(f"  TP={r['TP']:,}  FP={r['FP']}  TN={r['TN']:,}  FN={r['FN']}")
print(f"  FAR        : {r['FAR']:.2e}   ({r['FAR']*1e6:.3f} ppm)")
print(f"  MDR        : {r['MDR']:.2e}   ({r['MDR']*1e6:.3f} ppm)")
print(f"  Fidelity   : {r['Fidelity']:.8f}  ({r['Fidelity']*100:.6f} %)")
print(f"  L2 routing : {r['L2_routing_rate']*100:.4f} %")
print(f"  Erasure rate: {r['erasure_rate']*100:.4f} %")
print(f"  MF SNR     : {r['mf_snr']:.1f}")
print(f"  Speed      : {r['fps']:.1f} fps  ({r['ms_per_frame']:.1f} ms/frame)")
print("="*56)

out = pathlib.Path(base)/"results"/"benchmark.json"
with open(out,"w") as f: json.dump(r, f, indent=2)
print(f"\nSaved → {out}")


layer1_mf.py v2 written (prev-frame CUSUM gating)
mf_detector.py overflow fix applied
Calibrating … 

mu_atom=1100.6±14.0  mu_bg=63.8±10.5  SNR=84.8



  MF-Array Two-Layer System — Benchmark (PA-3HDA-MF v1.0)
  Sites=400  Frames=500  Decisions=200,000
  TP=46,955  FP=0  TN=153,045  FN=0
  FAR        : 0.00e+00   (0.000 ppm)
  MDR        : 0.00e+00   (0.000 ppm)
  Fidelity   : 1.00000000  (100.000000 %)
  L2 routing : 0.0000 %
  Erasure rate: 0.2250 %
  MF SNR     : 84.8
  Speed      : 10.3 fps  (97.4 ms/frame)

Saved → /data/workspace/61448ef7-eb83-45fe-a53b-e87021227cc1/mf_atom_detection/results/benchmark.json


In [23]:

import json, pathlib, subprocess, os

base = "/data/workspace/61448ef7-eb83-45fe-a53b-e87021227cc1/mf_atom_detection"
r = json.load(open(f"{base}/results/benchmark.json"))

readme = f"""# mf-atom-detection

**PA-3HDA-MF v1.0** — Matched Filter Array two-layer atom detection system for neutral atom quantum computing.

Replaces the L1 / L2 lightweight CNNs from PA-3HDA v10.0 with a **Matched Filter (MF) Array**.
L1.5 physical-diagnostic router is removed. Architecture reduced to two active detection layers.

---

## Architecture

```
Frame (short exp. {r['l1_exposure_ms']} ms)
  └─► L1: MF-Array + Bayesian temporal fusion + CUSUM
        ├─► ATOM_PRESENT   (p_post > θ_H = {r['l1_theta_H']})
        ├─► ATOM_ABSENT    (p_post < θ_L = {r['l1_theta_L']})
        ├─► ERASURE_LOSS   (CUSUM threshold crossed)
        └─► ROUTE_L2

Frame (long  exp. {r['l2_exposure_ms']} ms)
  └─► L2: MF-Array precise judgment
        ├─► ATOM_PRESENT
        └─► ATOM_ABSENT
```

### Why Matched Filter?

The Matched Filter is the SNR-optimal linear detector for a known signal in additive
white Gaussian noise (Neyman-Pearson lemma). For a Gaussian PSF atom signal:

```
score_k  = dot(roi_k.flat, h_k.flat)   # h_k: unit-norm Gaussian template
LLR(s_k) = -0.5*((s-mu_atom)/sigma_atom)^2 + 0.5*((s-mu_bg)/sigma_bg)^2 + log(sigma_bg/sigma_atom)
p_mf(k)  = sigmoid(LLR)               # calibrated posterior probability
```

Advantages over CNN:
- **Zero learned parameters** — no training required, no overfitting
- **Theoretically optimal** for the physical Poisson+Gaussian noise model
- **Pure dot-product** — faster inference, FPGA-friendly (DSP blocks)
- **Analytical performance bound** — SNR fully determines FAR/MDR at calibration time

### Bayesian Temporal Fusion (retained from PA-3HDA)

```
x(k,t) = x(k,t-1) + log(p_s/(1-p_s)) + λ · log(p_mf/(1-p_mf))
p_post  = sigmoid(x(k,t))
```

- p_s = {r.get('p_survival', 0.9952)} (atom survival probability per frame)
- λ = observation weight (default 0.8)
- Warmup: 5 frames (single-frame fallback)

### CUSUM Erasure Detection (retained, with gating fix)

```
S(k,t) = max(0, S(k,t-1) + log((1-p_mf)/p_mf)) · gate(prev_log_odds > 0)
```

Key fix vs PA-3HDA: CUSUM only accumulates when **previous frame** believed atom present
(`prev_log_odds > 0`). Prevents false ERASURE for perpetually empty sites.

---

## Benchmark Results

Config: 20×20 array (400 sites), 500 frames, simulated Yb-171 platform.

| Metric | Value |
|--------|-------|
| **Fidelity** | **{r['Fidelity']*100:.6f} %** |
| FAR (False Alarm Rate) | {r['FAR']:.2e} ({r['FAR']*1e6:.3f} ppm) |
| MDR (Miss Detection Rate) | {r['MDR']:.2e} ({r['MDR']*1e6:.3f} ppm) |
| TP / FP / TN / FN | {r['TP']:,} / {r['FP']} / {r['TN']:,} / {r['FN']} |
| L2 routing rate | {r['L2_routing_rate']*100:.4f} % |
| Erasure rate | {r['erasure_rate']*100:.4f} % |
| MF SNR | {r['mf_snr']:.1f} |
| Total decisions | {r['total_decisions']:,} |
| Speed | {r['fps']:.1f} fps ({r['ms_per_frame']:.1f} ms/frame, pure Python) |

MF calibration: mu_atom={r['mf_mu_atom']:.1f} ± {r['mf_sig_atom']:.1f},
mu_bg={r['mf_mu_bg']:.1f} ± {r['mf_sig_bg']:.1f}.

---

## Repository Structure

```
mf_atom_detection/
├── src/
│   ├── config.py        System-wide parameters (dataclasses)
│   ├── psf.py           Gaussian PSF model, unit-norm template generation
│   ├── mf_detector.py   Vectorized MF score + LRT probability conversion
│   ├── layer1_mf.py     L1: MF-Array + Bayesian log-odds fusion + CUSUM
│   ├── layer2_mf.py     L2: MF-Array precise judgment
│   ├── simulate.py      Synthetic Yb-171 fluorescence image simulator
│   └── system.py        Two-layer system orchestration + 5-type output
├── eval/
│   └── benchmark.py     FAR / MDR / Fidelity / routing-rate / speed
├── results/
│   └── benchmark.json   Latest evaluation results
└── README.md
```

## Quick Start

```bash
pip install numpy
python eval/benchmark.py
```

---

## Physical Parameters (default)

| Parameter | Value | Source |
|-----------|-------|--------|
| Photon rate | 35,000 ph/s/atom | ¹⁷¹Yb 399 nm imaging line |
| Camera QE | 72 % | Qbit 4610 sCMOS |
| Readout noise | 0.3 e⁻ | sCMOS spec |
| PSF sigma | 4.6 px | NA=0.5 objective |
| Site pitch | 54 px | 5 µm trap spacing |
| Background | 800 ph/s/site | |
| Crosstalk | 1 % | PSF tail leakage |
| t_L1 | {r['l1_exposure_ms']} ms | ~128 photons/atom |
| t_L2 | {r['l2_exposure_ms']} ms | independently calibrated |
| p_survival | 0.9952 | 0.48 %/frame loss rate |

---

## Five-Type Structured Output

| # | Output | Source | QEC Value |
|---|--------|--------|-----------|
| 1 | `ATOM_PRESENT` | L1 / L2 | — |
| 2 | `ATOM_ABSENT` | L1 / L2 | — |
| 3 | `ERASURE_LOSS` | L1 CUSUM | Erasure error → QEC threshold ~50 % |
| 4 | `CORR_LOSS` | reserved (L1.5 removed) | Correlated erasure |
| 5 | `DRIFT_ALARM` | L0 background (stub) | Triggers recalibration |

---

## Reference

PA-3HDA v10.0 patent disclosure — Hierarchical Adaptive Fluorescence Detection
System and Method for Neutral Atom Arrays in Quantum Computing (2026).
"""

with open(f"{base}/README.md", "w") as f:
    f.write(readme)
print("README updated with real benchmark numbers")
print(f"  Fidelity={r['Fidelity']*100:.6f}%  FAR={r['FAR']:.2e}  L2={r['L2_routing_rate']*100:.4f}%  Erasure={r['erasure_rate']*100:.4f}%")


README updated with real benchmark numbers
  Fidelity=100.000000%  FAR=0.00e+00  L2=0.0000%  Erasure=0.2250%
